# B21 BSA dilution series with server-side HDF5 chunks

This notebook runs the first processing stage of the Diamond Light Source B21 example. It inventories the donated batch, validates the relative Eiger links without loading full detector images, processes a representative sample/background subset in 21-frame chunk plans, and applies complete-measurement frame-quality flags.

The first pass uses the tracked B21 pre-filter integration pipeline. Each changing measurement is registered as an HDF source and its detector frames will be sliced inside the runtime; the notebook will not upload image arrays through a buffer. The resulting 50-bin curves must be assembled for a complete measurement before the B21 quality module chooses reference extrema and assigns flags.


## Configuration

Run from the examples repository root or from this directory. The default five-frame chunks split every 21-frame acquisition into four full chunks and one one-frame edge chunk.


In [ ]:
from pathlib import Path
import sys

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "example_utils.py").is_file():
        EXAMPLES_ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Start Jupyter from MoDaCor_examples or one of its subdirectories."
    )

sys.path.insert(0, str(EXAMPLES_ROOT))
from example_utils import locate_example_dir

PROJECT_DIR = locate_example_dir("DLS/B21")
sys.path.insert(0, str(PROJECT_DIR))
DATA_DIR = PROJECT_DIR / "data"
WORK_DIR = PROJECT_DIR / "work" / "chunk_server"
PIPELINE_PATH = PROJECT_DIR / "pipelines" / "B21_frame_prefilter.yaml"
QUALITY_PIPELINE_PATH = PROJECT_DIR / "pipelines" / "B21_frame_quality.yaml"
PREFILTER_OUTPUT_DIR = WORK_DIR / "prefilter"
QUALITY_OUTPUT_DIR = WORK_DIR / "quality"
CALIBRATION_PATH = DATA_DIR / "processing" / "calibration_file_140926.nxs"
MASK_PATH = DATA_DIR / "processing" / "mask_file_140926.nxs"


In [ ]:
import atexit

import h5py
import hdf5plugin
import matplotlib.pyplot as plt
import modacor
import numpy as np
from IPython.display import JSON, Markdown, display
from matplotlib.colors import LogNorm
from modacor.client import LocalRuntimeServer
from modacor.runner.pipeline import Pipeline

from b21_helpers import (
    compare_buffer_measurements,
    load_quality_arrays,
    process_prefilter_measurement,
    process_quality_measurement,
)

DETECTOR_PATH = "/entry1/instrument/detector/data"
MASK_DATASET_PATH = "/entry/mask/mask"
DETECTOR_VALID_MIN = 0
DETECTOR_VALID_MAX = 4294967293
CHUNK_SIZE = 5
LOW_Q_MAX = 0.02
HIGH_Q_MIN = 0.15
MAX_ACCEPTED_EXAMPLES = 2
MAX_REJECTED_EXAMPLES = 3
DISPLAY_STRIDE = 4
BUFFER_RELATIVE_DISTANCE_LIMIT = 0.01
SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8901
EXPECTED_TITLES = (
    "buffer_1",
    "bsa_10mgml",
    "bsa_5mgml",
    "bsa_2p5mgml",
    "bsa_1p25mgml",
    "bsa_0p6mgml",
    "bsa_0p3mgml",
    "buffer_2",
)
PROCESS_TITLES = (
    "buffer_1",
    "bsa_10mgml",
    "bsa_2p5mgml",
    "bsa_0p3mgml",
    "buffer_2",
)


## Pre-filter pipeline graphs

The coarse azimuthal average is a side branch: `sample` retains the detector frames, while `prefilter` receives shared, read-only references to the signal, mask, geometry, and pixel index. The first graph stops after producing one 50-bin curve per frame. The compact second graph runs only after those curves have been assembled for a complete measurement, ensuring the B21 quality references are not selected independently inside each five-frame chunk.


In [ ]:
for heading, pipeline_path in (
    ("Chunked detector integration", PIPELINE_PATH),
    ("Complete-measurement quality pass", QUALITY_PIPELINE_PATH),
):
    prepared_pipeline = Pipeline.from_yaml_file(yaml_file=pipeline_path)
    prepared_pipeline.prepare()
    display(
        Markdown(
            f"### {heading}\n\n```mermaid\n"
            f"{prepared_pipeline.to_mermaid(direction='TD')}\n```"
        )
    )
    print(f"Prepared {len(prepared_pipeline.graph)} steps from {pipeline_path.name}.")


## Discover and validate the raw batch

Only metadata, link targets, shapes, and dtypes are inspected here. Accessing the detector dataset resolves the relative NeXus-to-Eiger link but does not materialize its image stack.


In [ ]:
def text_value(dataset):
    value = dataset.asstr()[()]
    return str(value.item() if getattr(value, "shape", None) == () else value)


runs = []
for master_path in sorted(DATA_DIR.glob("b21-*.nxs")):
    with h5py.File(master_path, "r") as nexus:
        title = text_value(nexus["/entry1/title"])
        detector = nexus[DETECTOR_PATH]
        detector_group = nexus["/entry1/instrument/detector"]
        link = detector_group.get("data", getlink=True)
        if not isinstance(link, h5py.ExternalLink):
            raise TypeError(f"{master_path.name}: detector data is not an external link")
        target = master_path.parent / link.filename
        if not target.is_file():
            raise FileNotFoundError(f"{master_path.name}: missing {link.filename}")
        runs.append(
            {
                "run": master_path.stem,
                "title": title,
                "role": "buffer" if title.startswith("buffer") else "sample",
                "master_path": master_path,
                "detector_shape": tuple(detector.shape),
                "detector_dtype": str(detector.dtype),
                "eiger_target": target.name,
            }
        )

titles = tuple(run["title"] for run in runs)
if titles != EXPECTED_TITLES:
    raise ValueError(f"Unexpected B21 run order or titles: {titles}")
if {run["detector_shape"][:2] for run in runs} != {(1, 21)}:
    raise ValueError("Expected one 21-frame acquisition in every raw master.")

display(
    JSON(
        [
            {key: value for key, value in run.items() if key != "master_path"}
            for run in runs
        ]
    )
)


## Prepare direct-HDF source registrations and chunk schedule

The runtime session uses direct HDF sources for the changing sample plus the fixed calibration and mask. Each provisional chunk plan binds `sample` plus `DETECTOR_PATH` as its driver and resolves selections of the form `(all, start:stop, all, all)`. The calibration and instrument mask remain static; a frame-wise threshold mask additionally rejects signal values below zero or above `4294967293`, covering the Eiger max-pegged sentinels.


In [ ]:
def hdf_source_registration(master_path):
    return {
        "ref": "sample",
        "type": "hdf",
        "location": str(Path(master_path).resolve()),
    }


static_sources = (
    {"ref": "calibration", "type": "hdf", "location": str(CALIBRATION_PATH.resolve())},
    {"ref": "mask", "type": "hdf", "location": str(MASK_PATH.resolve())},
)
for required in (PIPELINE_PATH, QUALITY_PIPELINE_PATH, CALIBRATION_PATH, MASK_PATH):
    if not required.is_file():
        raise FileNotFoundError(required)

work_items = []
for measurement_index, run in enumerate(runs):
    frame_count = run["detector_shape"][1]
    for chunk_index, start in enumerate(range(0, frame_count, CHUNK_SIZE)):
        stop = min(start + CHUNK_SIZE, frame_count)
        work_items.append(
            {
                "chunk_id": f"m{measurement_index:03d}-c{chunk_index:03d}",
                "run": run["run"],
                "title": run["title"],
                "start": start,
                "stop": stop,
                "input_shape": (1, stop - start, *run["detector_shape"][2:]),
                "source": hdf_source_registration(run["master_path"]),
            }
        )

print(f"Python: {sys.executable}")
print(f"MoDaCor: {modacor.__version__}")
print(f"Measurements: {len(runs)}")
print(f"Chunks: {len(work_items)} ({len(work_items) // len(runs)} per measurement)")
display(JSON(work_items[:5]))


## Start or reuse the MoDaCor runtime

This establishes the server boundary and prepares a reusable session with the tracked integration pipeline and direct HDF sources. The fixed geometry and mask branch is calculated on the first chunk and reused while the changing detector source is sliced frame-wise by the server.


In [ ]:
WORK_DIR.mkdir(parents=True, exist_ok=True)
server = LocalRuntimeServer(
    host=SERVER_HOST,
    port=SERVER_PORT,
    log_path=WORK_DIR / "modacor_server.log",
    environment={"HDF5_PLUGIN_PATH": hdf5plugin.PLUGINS_PATH},
)
client = server.start()
atexit.register(server.stop)
print(f"{'Started' if server.launched else 'Reusing'} runtime at {client.base_url}")

session = client.replace_session(
    "b21-frame-prefilter-hdf",
    name="B21 frame pre-filter direct-HDF chunks",
    pipeline_yaml=PIPELINE_PATH.read_text(encoding="utf-8"),
)
session.register_sources(*static_sources, work_items[0]["source"])
print(f"Prepared session {session.session_id} with direct HDF sources.")


## Run representative samples and both buffer backgrounds

The initial processing set contains the two bracketing buffers plus high-, intermediate-, and low-concentration BSA samples. Each raw run is read in five server-side chunks and assembled into a compact HDF5 pre-filter product. A second server session then applies `B21FrameQualityFilter` once to each complete 21-frame curve set. Completed generated products are reused on notebook reruns; an incomplete existing work file is never overwritten automatically.

These are quality-assessment products, not yet final corrected SAXS curves. The later workflow still needs decisions about interpolation or selection of the two buffers, incident-flux and integrating beamstop-monitor normalization, exposure time, accepted-frame averaging, subtraction, and DAWN comparison.


In [ ]:
run_by_title = {run["title"]: run for run in runs}
selected_runs = [run_by_title[title] for title in PROCESS_TITLES]

prefilter_results = []
prefilter_session_has_state = False
for run in selected_runs:
    output_path = PREFILTER_OUTPUT_DIR / f"{run['run']}_{run['title']}.h5"
    result = process_prefilter_measurement(
        client=client,
        session=session,
        run=run,
        output_path=output_path,
        chunk_size=CHUNK_SIZE,
        first_mode="partial" if prefilter_session_has_state else "full",
    )
    prefilter_results.append(result)
    if not result["cached"]:
        prefilter_session_has_state = True
    state = "cached" if result["cached"] else f"{result['chunks']} chunks"
    print(f"{result['title']}: pre-filter curves ready ({state})")

quality_session = client.replace_session(
    "b21-frame-quality",
    name="B21 complete-measurement frame quality",
    pipeline_yaml=QUALITY_PIPELINE_PATH.read_text(encoding="utf-8"),
)
quality_results = []
for run, prefilter_result in zip(selected_runs, prefilter_results, strict=True):
    output_path = QUALITY_OUTPUT_DIR / f"{run['run']}_{run['title']}.h5"
    result = process_quality_measurement(
        session=quality_session,
        run=run,
        prefilter_path=prefilter_result["path"],
        output_path=output_path,
    )
    quality_results.append(result)

display(
    JSON(
        [
            {key: str(value) if isinstance(value, Path) else value for key, value in result.items()}
            for result in quality_results
        ]
    )
)


## Initial/final buffer equivalence

This check uses only frames accepted by the B21 quality filter. Its primary practical metric is the RMS log ratio between the final- and initial-buffer mean curves; `expm1` converts that distance back to a fractional change, which is compared with the provisional 1% limit. The scale-adjusted distance separately tests curve shape after removing one multiplicative factor. A combined-SEM z-distance is also reported, because two very stable measurements can be statistically distinguishable while remaining comfortably inside a practical 1% tolerance.

This comparison currently precedes the agreed incident-flux and beamstop-monitor normalization, so it is a prefilter stability check rather than the final background-acceptance decision.


In [ ]:
quality_by_title = {result["title"]: result for result in quality_results}
buffer_comparison = compare_buffer_measurements(
    quality_by_title["buffer_1"]["path"],
    quality_by_title["buffer_2"]["path"],
    relative_limit=BUFFER_RELATIVE_DISTANCE_LIMIT,
)
buffer_metrics = buffer_comparison["metrics"]
display(JSON(buffer_metrics))

q = buffer_comparison["q"]
valid = buffer_comparison["valid"]
figure, (curve_ax, difference_ax) = plt.subplots(
    2, 1, figsize=(10, 8), sharex=True, constrained_layout=True
)
for label, colour, mean_key, sem_key in (
    ("initial buffer", "tab:blue", "initial_mean", "initial_sem"),
    ("final buffer", "tab:orange", "final_mean", "final_sem"),
):
    mean = buffer_comparison[mean_key]
    sem = buffer_comparison[sem_key]
    curve_ax.plot(q[valid], mean[valid], color=colour, label=label)
    curve_ax.fill_between(
        q[valid],
        np.maximum(mean[valid] - sem[valid], np.finfo(float).tiny),
        mean[valid] + sem[valid],
        color=colour,
        alpha=0.2,
    )
curve_ax.set(xscale="log", yscale="log", ylabel="mean coarse I(q)")
curve_ax.legend()
curve_ax.grid(alpha=0.2)

difference_percent = 100.0 * buffer_comparison["symmetric_relative_difference"]
limit_percent = 100.0 * BUFFER_RELATIVE_DISTANCE_LIMIT
difference_ax.axhspan(-limit_percent, limit_percent, color="tab:green", alpha=0.12)
difference_ax.axhline(0.0, color="black", linewidth=0.8)
difference_ax.plot(q[valid], difference_percent[valid], color="tab:purple")
difference_ax.set(
    xscale="log",
    xlabel=r"$q$ / $\AA^{-1}$",
    ylabel="symmetric difference / %",
)
difference_ax.grid(alpha=0.2)
status = "PASS" if buffer_metrics["within_relative_limit"] else "FAIL"
figure.suptitle(
    f"Initial/final buffer: {status}; RMS distance "
    f"{100 * buffer_metrics['curve_log_rms_distance']:.3f}% "
    f"(limit {limit_percent:.1f}%)"
)
plt.show()


## Accepted and rejected frame diagnostics

After the chunk curves for one measurement have been assembled and `B21FrameQualityFilter` has assigned flags, use the helper below to inspect a bounded set of decisions. It chooses accepted frames across their acquisition span and prioritizes rejected frames with distinct bit patterns. Each row shows the coarse 1D curve and the corresponding 2D frame at the end of the current prefilter: the fixed instrument mask and lower/upper detector threshold have been applied before display. Only the selected source frame is read, transformed, and strided; the full detector stack is never loaded into the notebook.

Pass the complete-measurement arrays as `coarse_signal`, `coarse_q`, and `flags`. Flag values are `0` (accepted), `1` (high-q total too low), `2` (low-q total too high), and `3` (both tests failed). The shaded 1D regions correspond to the provisional B21 thresholds.


In [ ]:
def _spread_indices(indices, limit):
    indices = np.asarray(indices, dtype=int)
    if limit <= 0 or indices.size == 0:
        return []
    positions = np.linspace(0, indices.size - 1, min(limit, indices.size))
    return [int(indices[round(position)]) for position in positions]


def _representative_indices(flags, max_accepted, max_rejected):
    flat_flags = np.asarray(flags, dtype=np.uint32).reshape(-1)
    accepted = _spread_indices(np.flatnonzero(flat_flags == 0), max_accepted)

    rejected = []
    for flag_value in (1, 2, 3):
        candidates = np.flatnonzero(flat_flags == flag_value)
        if candidates.size and len(rejected) < max_rejected:
            rejected.append(int(candidates[candidates.size // 2]))
    remaining = np.setdiff1d(np.flatnonzero(flat_flags != 0), rejected)
    rejected.extend(_spread_indices(remaining, max_rejected - len(rejected)))
    return accepted + rejected


def _flag_description(flag):
    if flag == 0:
        return "accepted"
    reasons = []
    if flag & 1:
        reasons.append("high-q low")
    if flag & 2:
        reasons.append("low-q high")
    return " + ".join(reasons)


def plot_frame_examples(
    master_path,
    coarse_signal,
    coarse_q,
    flags,
    *,
    max_accepted=MAX_ACCEPTED_EXAMPLES,
    max_rejected=MAX_REJECTED_EXAMPLES,
    display_stride=DISPLAY_STRIDE,
    mask_path=MASK_PATH,
):
    curves = np.asarray(coarse_signal, dtype=float)
    if curves.ndim < 2:
        raise ValueError("coarse_signal must contain a frame and a q-bin dimension")
    curves = curves.reshape(-1, curves.shape[-1])
    q_values = np.broadcast_to(np.asarray(coarse_q, dtype=float), np.shape(coarse_signal))
    q_values = q_values.reshape(curves.shape)
    flat_flags = np.asarray(flags, dtype=np.uint32).reshape(-1)
    if flat_flags.size != curves.shape[0]:
        raise ValueError("flags and coarse_signal describe different frame counts")

    selected = _representative_indices(flat_flags, max_accepted, max_rejected)
    if not selected:
        raise ValueError("No accepted or rejected frames are available to plot")

    master_path = Path(master_path)
    figure, axes = plt.subplots(
        len(selected),
        2,
        figsize=(12, 3.4 * len(selected)),
        squeeze=False,
        constrained_layout=True,
    )
    with h5py.File(mask_path, "r") as mask_file:
        instrument_mask = np.asarray(mask_file[MASK_DATASET_PATH])[
            ::display_stride, ::display_stride
        ] != 0

    with h5py.File(master_path, "r") as nexus:
        detector = nexus[DETECTOR_PATH]
        raw_frame_shape = detector.shape[:-2]
        if int(np.prod(raw_frame_shape)) != curves.shape[0]:
            raise ValueError(
                f"{master_path.name} has {raw_frame_shape} frame axes, but the curves "
                f"describe {curves.shape[0]} frames"
            )

        for row, flat_index in enumerate(selected):
            flag = int(flat_flags[flat_index])
            colour = "tab:green" if flag == 0 else "tab:red"
            curve_ax, image_ax = axes[row]

            finite_positive = (q_values[flat_index] > 0) & (curves[flat_index] > 0)
            curve_ax.plot(
                q_values[flat_index, finite_positive],
                curves[flat_index, finite_positive],
                color=colour,
            )
            curve_ax.axvspan(
                np.nanmin(q_values[flat_index]), LOW_Q_MAX, color="tab:orange", alpha=0.15
            )
            curve_ax.axvspan(
                HIGH_Q_MIN, np.nanmax(q_values[flat_index]), color="tab:blue", alpha=0.12
            )
            curve_ax.set(
                xscale="log", yscale="log", xlabel=r"$q$ / $\AA^{-1}$", ylabel="coarse I(q)"
            )
            curve_ax.set_title(
                f"Frame {flat_index}: {_flag_description(flag)} (flag {flag})", color=colour
            )
            curve_ax.grid(alpha=0.2)

            raw_index = tuple(int(value) for value in np.unravel_index(flat_index, raw_frame_shape))
            frame = np.asarray(
                detector[raw_index][::display_stride, ::display_stride], dtype=float
            )
            threshold_mask = (frame < DETECTOR_VALID_MIN) | (frame > DETECTOR_VALID_MAX)
            frame[instrument_mask | threshold_mask] = np.nan
            positive = frame[np.isfinite(frame) & (frame > 0)]
            if positive.size:
                vmin, vmax = np.percentile(positive, (1.0, 99.7))
                if vmax <= vmin:
                    vmax = vmin + 1.0
                image = image_ax.imshow(
                    frame,
                    cmap="magma",
                    norm=LogNorm(vmin=max(float(vmin), 1.0), vmax=float(vmax)),
                )
            else:
                image = image_ax.imshow(frame, cmap="magma")
            image_ax.set_title(
                f"End-of-prefilter 2D frame {raw_index} (stride {display_stride})"
            )
            image_ax.set_axis_off()
            figure.colorbar(
                image, ax=image_ax, fraction=0.046, pad=0.02, label="detector count"
            )

    figure.suptitle(master_path.stem)
    return figure, selected


In [ ]:
diagnostic_selections = {}
for run, result in zip(selected_runs, quality_results, strict=True):
    arrays = load_quality_arrays(result["path"])
    figure, selected_frames = plot_frame_examples(
        run["master_path"],
        coarse_signal=arrays["signal"],
        coarse_q=arrays["Q"],
        flags=arrays["flags"],
    )
    figure.suptitle(f"{run['title']} ({run['run']})")
    diagnostic_selections[run["title"]] = selected_frames
    plt.show()

display(JSON(diagnostic_selections))


## Cleanup


In [ ]:
quality_session.delete()
session.delete()
server.stop()
print(
    "Stopped the notebook-owned runtime."
    if not client.is_ready()
    else "Left the external runtime running."
)
